# Notebook 18 - Extended Evaluation Metrics (AUC-ROC, PR-AUC, ECE, PFI)

**Abhishek Tomar | PN1196973 | LJMU | April 2026**

---

Fills the metric gaps the proposal/master document promises but the earlier notebooks
did not compute:

- **AUC-ROC** and **PR-AUC** (macro, one-vs-rest) per model and per subgroup —
  threshold-independent discrimination, which macro F1 cannot show.
- **ECE (Expected Calibration Error)** per subgroup — shows *where* in the probability
  range the model is miscalibrated, complementing the Brier score already computed.
- **PFI (Permutation Feature Importance)** for the final model, cross-checked against
  SHAP rankings via Jaccard overlap — independent validation of the XAI findings.

Cells 2-4 are pure post-processing on saved probability files (no training). Cell 5
(PFI) reloads the final model's features and refits a lightweight model for permutation.


## Cell 1: Setup

In [1]:
!pip install -q scikit-learn pandas pyarrow
from google.colab import drive; drive.mount('/content/drive')
import sys; sys.path.append(r"/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation")
from thesis_utils import *
import pandas as pd, numpy as np
from sklearn.metrics import roc_auc_score, average_precision_score
from sklearn.preprocessing import label_binarize
CLASSES=["negative","neutral","positive"]
SUBS=["emoji-heavy","slang-heavy","sarcasm","formal"]
OUT=PATHS["results"]/"assembled"; OUT.mkdir(parents=True,exist_ok=True)
def proba(df):
    cols=sorted(c for c in df.columns if c.startswith("proba_"))
    return df[cols].values
print("ready")

Mounted at /content/drive
thesis_utils loaded. [V2 FIXED: Fairlearn EOD, Theil index]
  seed = 42
  subgroup thresholds: emoji > 0.05, slang > 0.1, formal min words = 8
  reference subgroup  = formal
ready


## Cell 2: AUC-ROC and PR-AUC per model (macro, one-vs-rest)

Uses the saved probability columns. AUC and PR-AUC are threshold-independent, so they
reveal discrimination ability that a single-threshold macro F1 hides — directly
addressing the proposal's note that threshold-dependent metrics can mask bias.

In [2]:
MODELS=[("exp01","LogisticRegression"),("exp02","LinearSVM"),("exp03","MultinomialNB"),
        ("exp04","RandomForest"),("exp05","XGBoost")]
rows=[]
for eid,name in MODELS:
    d=load_predictions(eid,name); P=proba(d)
    yt=d["y_true"].values
    Yb=label_binarize(yt,classes=CLASSES)
    try:
        roc=roc_auc_score(Yb,P,average="macro",multi_class="ovr")
    except Exception: roc=np.nan
    prauc=average_precision_score(Yb,P,average="macro")
    # per-class
    row={"Model":name,"ROC-AUC (macro OvR)":round(roc,4),"PR-AUC (macro)":round(prauc,4)}
    for i,c in enumerate(CLASSES):
        row[f"ROC-AUC {c}"]=round(roc_auc_score(Yb[:,i],P[:,i]),4)
        row[f"PR-AUC {c}"]=round(average_precision_score(Yb[:,i],P[:,i]),4)
    rows.append(row)
auc_tbl=pd.DataFrame(rows).sort_values("ROC-AUC (macro OvR)",ascending=False)
auc_tbl.to_csv(OUT/"TableX1_AUC_PRAUC.csv",index=False)
print(auc_tbl.to_string(index=False))
print("\nNote: negative-class PR-AUC is the key number — it shows minority-class ranking quality even where recall is low.")

             Model  ROC-AUC (macro OvR)  PR-AUC (macro)  ROC-AUC negative  PR-AUC negative  ROC-AUC neutral  PR-AUC neutral  ROC-AUC positive  PR-AUC positive
         LinearSVM               0.7633          0.6264            0.7800           0.6318           0.6742          0.6273            0.8358           0.6201
LogisticRegression               0.7575          0.6189            0.7744           0.6192           0.6692          0.6262            0.8290           0.6113
     MultinomialNB               0.7545          0.6085            0.7705           0.6030           0.6590          0.6140            0.8339           0.6084
           XGBoost               0.7299          0.5788            0.7350           0.5821           0.6673          0.6093            0.7874           0.5449
      RandomForest               0.7291          0.5833            0.7414           0.5607           0.6690          0.6315            0.7768           0.5576

Note: negative-class PR-AUC is the key number

## Cell 3: ECE (Expected Calibration Error) per subgroup - final model

Bins predictions by confidence and measures the gap between confidence and accuracy in
each bin. Shows whether the model is systematically over/under-confident, and where -
the master document notes sarcasm/slang posts cluster in the uncertain 0.4-0.6 range.

In [3]:
def ece(y_true,y_pred,conf,n_bins=10):
    y_true=np.asarray(y_true);y_pred=np.asarray(y_pred);conf=np.asarray(conf)
    correct=(y_true==y_pred).astype(float)
    bins=np.linspace(0,1,n_bins+1); e=0.0; N=len(conf); detail=[]
    for i in range(n_bins):
        m=(conf>bins[i])&(conf<=bins[i+1])
        if m.sum()==0: continue
        acc=correct[m].mean(); cf=conf[m].mean(); w=m.sum()/N
        e+=w*abs(acc-cf); detail.append((round(bins[i],1),int(m.sum()),round(cf,3),round(acc,3)))
    return e,detail

pred=load_predictions("exp10","FinalModel")
P=proba(pred); conf=P.max(1)
rows=[]
for sg in SUBS:
    part=pred[pred["subgroup"]==sg]
    if len(part)==0: continue
    Pp=proba(part); cf=Pp.max(1)
    e,_=ece(part["y_true"].values,part["y_pred"].values,cf)
    rows.append({"Subgroup":sg,"N":len(part),"ECE":round(e,4),
                 "Mean Confidence":round(cf.mean(),4),
                 "Accuracy":round((part["y_true"].values==part["y_pred"].values).mean(),4)})
overall_e,_=ece(pred["y_true"].values,pred["y_pred"].values,conf)
ece_tbl=pd.DataFrame(rows)
ece_tbl.to_csv(OUT/"TableX2_ECE_by_subgroup.csv",index=False)
print(f"Overall ECE: {overall_e:.4f}\n")
print(ece_tbl.to_string(index=False))
print("\nHigher ECE = worse calibration. Compare against Brier (Table 1) and HCER (Table 10).")

Overall ECE: 0.1301

   Subgroup     N    ECE  Mean Confidence  Accuracy
emoji-heavy   696 0.1334           0.7531    0.6207
slang-heavy    60 0.1315           0.7482    0.6167
    sarcasm    14 0.1660           0.7196    0.7857
     formal 10398 0.1314           0.7234    0.5920

Higher ECE = worse calibration. Compare against Brier (Table 1) and HCER (Table 10).


## Cell 4: AUC and PR-AUC per subgroup final model

Same discrimination metrics, now sliced by linguistic subgroup, so you can state whether
the model's ranking ability itself degrades on informal language (not just its thresholded F1).

In [4]:
rows=[]
for sg in SUBS:
    part=pred[pred["subgroup"]==sg]
    if len(part)<5: continue
    yt=part["y_true"].values; Pp=proba(part); Yb=label_binarize(yt,classes=CLASSES)
    # guard: subgroup may lack a class
    try: roc=roc_auc_score(Yb,Pp,average="macro",multi_class="ovr")
    except Exception: roc=np.nan
    try: pr=average_precision_score(Yb,Pp,average="macro")
    except Exception: pr=np.nan
    rows.append({"Subgroup":sg,"N":len(part),"ROC-AUC (macro)":round(roc,4) if roc==roc else "n/a",
                 "PR-AUC (macro)":round(pr,4) if pr==pr else "n/a"})
sub_auc=pd.DataFrame(rows)
sub_auc.to_csv(OUT/"TableX3_AUC_by_subgroup.csv",index=False)
print(sub_auc.to_string(index=False))

   Subgroup     N  ROC-AUC (macro)  PR-AUC (macro)
emoji-heavy   696           0.7882          0.6423
slang-heavy    60           0.8067          0.7101
    sarcasm    14           0.9512          0.9461
     formal 10398           0.7724          0.6352


## Cell 5: PFI (Permutation Feature Importance) vs SHAP-final model

Refits the final Logistic Regression on the saved hybrid features, computes permutation
importance, and compares its top features against the SHAP top features via Jaccard
overlap. Independent validation: if PFI and SHAP agree, the XAI finding is robust.

This cell trains one LR (fast). If features are unavailable it skips gracefully.

In [5]:
import scipy.sparse as sp
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
D=PATHS["data"]
try:
    Xtr=sp.load_npz(D/"tw_Xtrain_hybrid.npz"); Xte=sp.load_npz(D/"tw_Xtest_hybrid.npz")
    ytr=pd.read_parquet(D/"tw_train.parquet")["sentiment"].values
    yte=pd.read_parquet(D/"tw_test.parquet")["sentiment"].values
    feat_names=json.load(open(D/"hybrid_feature_names.json")) if (D/"hybrid_feature_names.json").exists() else None
    lr=LogisticRegression(C=10,penalty="l2",solver="liblinear",max_iter=2000).fit(Xtr,ytr)
    # permutation importance on a sample for speed (sparse -> use a subset of columns via dense sample)
    idx=np.random.RandomState(42).choice(Xte.shape[0],size=min(2000,Xte.shape[0]),replace=False)
    r=permutation_importance(lr,Xte[idx].toarray(),yte[idx],n_repeats=5,random_state=42,scoring="f1_macro")
    order=np.argsort(r.importances_mean)[::-1][:20]
    pfi_top=[(feat_names[i] if feat_names else f"f{i}", round(float(r.importances_mean[i]),5)) for i in order]
    pfi_tbl=pd.DataFrame(pfi_top,columns=["Feature","PFI Importance"])
    pfi_tbl.to_csv(OUT/"TableX4_PFI_top20.csv",index=False)
    print("Top-20 PFI features:"); print(pfi_tbl.to_string(index=False))
    print("\nCompare these against the SHAP top features (Table 6b). Overlap = robust; disagreement = model-specific.")
except FileNotFoundError as e:
    print("Hybrid feature matrix not found — skipping PFI. Missing:",e)
    print("If you want PFI, save tw_Xtrain_hybrid.npz / tw_Xtest_hybrid.npz + hybrid_feature_names.json from the feature-build notebook.")

Hybrid feature matrix not found — skipping PFI. Missing: [Errno 2] No such file or directory: '/content/drive/MyDrive/My_Data/upgrad-ljmu-thesis/LJMU Thesis/Implementation/data/tw_Xtrain_hybrid.npz'
If you want PFI, save tw_Xtrain_hybrid.npz / tw_Xtest_hybrid.npz + hybrid_feature_names.json from the feature-build notebook.


## Cell 6: Done

New CSVs in `results/assembled/`: TableX1 (AUC/PR-AUC per model), TableX2 (ECE per
subgroup), TableX3 (AUC per subgroup), TableX4 (PFI top-20). These fill the metric gaps
the master document lists (AUC-ROC, PR-AUC, ECE) and add PFI as independent SHAP validation.

In [6]:
for f in ["TableX1_AUC_PRAUC.csv","TableX2_ECE_by_subgroup.csv","TableX3_AUC_by_subgroup.csv","TableX4_PFI_top20.csv"]:
    p=OUT/f
    print(f"  {f}: {'OK' if p.exists() else 'skipped'}")

  TableX1_AUC_PRAUC.csv: OK
  TableX2_ECE_by_subgroup.csv: OK
  TableX3_AUC_by_subgroup.csv: OK
  TableX4_PFI_top20.csv: skipped
